<a href="https://colab.research.google.com/github/b-kaleem-5836668/Analytics-Engineering/blob/main/2_03_practice_bike_hire_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [9]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [10]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [11]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

In [12]:
# Inspect the first few rows of each table to understand their structure and content
print("--- Trips (Sample) ---")
display(q("SELECT * FROM trips "))

print("\n--- Bikes (Sample) ---")
display(q("SELECT * FROM bikes "))

print("\n--- Stations (Sample) ---")
display(q("SELECT * FROM stations "))

trips['user_type'].unique()

trips.nlargest(10, "duration_min")


--- Trips (Sample) ---


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member
...,...,...,...,...,...,...,...
607,T0596,B051,S02,S02,2025-03-31 13:51:00,NaN,Casual
608,T0597,B011,S01,S04,2025-03-31 14:00:00,19.0,Member
609,T0598,B046,S06,S11,2025-03-31 15:38:00,12.0,Member
610,T0599,B027,S04,S02,31/03/2025 18:38,21.0,casual



--- Bikes (Sample) ---


,bike_id,bike_type,purchase_year
0,B001,Standard,2021
1,B002,Electric,2022
2,B003,electric,2024
3,B004,Standard,2021
4,B005,STANDARD,2022
5,B006,Standard,2024
6,B007,standard,2024
7,B008,Standard,2021
8,B009,Standard,2021
9,B010,Standard,2023



--- Stations (Sample) ---


,station_id,station_name,zone,capacity
0,S01,Market Square,Centre,29.0
1,S02,Railway Station,Centre,23.0
2,S03,Canal Basin,North,24.0
3,S04,University Gate,West,28.0
4,S05,Park Road,South,22.0
5,S06,Library Corner,Centre,26.0
6,S07,Hospital Entrance,East,27.0
7,S08,Riverside Walk,South,14.0
8,S09,Old Town Hall,Centre,NaN
9,S10,Retail Park,North,16.0


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
80,T0081,B010,S12,S11,2025-03-04 21:57:00,2880.0,Member
143,T0143,B020,S09,S10,2025-03-08 12:23:00,1440.0,Casual
287,T0284,B046,S08,S03,2025-03-15 20:25:00,1440.0,Member
127,T0127,B025,S08,S09,2025-03-07 09:01:00,90.0,Member
357,T0352,B042,S03,S02,2025-03-19 08:59:00,67.0,Member
326,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member
327,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member
520,T0512,B045,S07,S04,26/03/2025 18:32,51.0,Member
37,T0038,B016,S12,S09,2025-03-02 12:23:00,50.0,Casual
262,T0260,B025,S04,S04,15/03/2025 07:29,46.0,Member


## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

In [16]:
# Clean and standardize bike_type in the bikes table
bikes = tables["bikes"].copy()

# Convert to lowercase and strip whitespace for consistency
bikes['bike_type_clean'] = bikes['bike_type'].str.lower().str.strip()

# Map variations to either 'electric' or 'standard'
def classify_bike(b_type):
    if any(x in b_type for x in ['electric', 'e-bike', 'ebike']):
        return 'electric'
    else:
        return 'standard'

bikes['bike_type_clean'] = bikes['bike_type_clean'].apply(classify_bike)

# Update our tables dictionary and register back to DuckDB
tables["bikes"] = bikes
con.register("bikes", bikes)

# Verify the cleaning results
print("Original categories:")
print(tables["bikes"]['bike_type'].value_counts())
print("\nCleaned and standardized categories:")
print(tables["bikes"]['bike_type_clean'].value_counts())


# Clean and standardize user_type in the trips table
trips = tables["trips"].copy()

# Convert to title case (e.g., 'casual' -> 'Casual', 'MEMBER' -> 'Member') and strip whitespace
trips['user_type_clean'] = trips['user_type'].str.strip().str.title()

# Map any remaining inconsistencies to ensure only 'Casual' and 'Member' are present
# (Title case automatically handles variations like 'Casual', 'CASUAL', 'casual', 'Member', 'MEMBER', 'member')

# Update our tables dictionary and register back to DuckDB
tables["trips"] = trips
con.register("trips", trips)

# Verify the cleaning results
print("Original categories:")
print(tables["trips"]['user_type'].value_counts(dropna=False))
print("\nCleaned and standardized categories:")
print(tables["trips"]['user_type_clean'].value_counts(dropna=False))

print (trips.duplicated().sum())



#The following code cleans the trips data and removes the twelve duplicates we just found abvove.

# @title
# Remove duplicate rows from the trips DataFrame
trips = tables["trips"].copy()

original_row_count = len(trips)
trips = trips.drop_duplicates()
cleaned_row_count = len(trips)

# Update our tables dictionary and register back to DuckDB
tables["trips"] = trips
con.register("trips", trips)

# Verify duplicate removal
print(f"Rows before removing duplicates: {original_row_count}")
print(f"Rows after removing duplicates: {cleaned_row_count}")
print(f"Number of remaining duplicates: {trips.duplicated().sum()}")










Original categories:
bike_type
Standard     36
Electric      9
STANDARD      8
standard      3
Electric      2
electric      1
E-bike        1
Name: count, dtype: int64

Cleaned and standardized categories:
bike_type_clean
standard    47
electric    13
Name: count, dtype: int64
Original categories:
user_type
Member     324
Casual     186
member      34
MEMBER      28
CASUAL      11
Casual       9
casual       8
Name: count, dtype: int64

Cleaned and standardized categories:
user_type_clean
Member    386
Casual    214
Name: count, dtype: int64
0
Rows before removing duplicates: 600
Rows after removing duplicates: 600
Number of remaining duplicates: 0


In [17]:
#Writes a code to identify outliers in the 'trips' table.

# 1. Summary Statistics of trip durations
duration_stats = tables["trips"]["duration_min"].describe()
print("--- Summary Statistics ---")
print(duration_stats)

# 2. Check extreme percentiles (e.g., 99th percentile)
p99 = tables["trips"]["duration_min"].quantile(0.99)
print(f"\n99th Percentile: {p99} minutes")

# 3. Use IQR method to define outlier bounds
Q1 = tables["trips"]["duration_min"].quantile(0.25)
Q3 = tables["trips"]["duration_min"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = tables["trips"][(tables["trips"]["duration_min"] < lower_bound) | (tables["trips"]["duration_min"] > upper_bound)]

print(f"\nIQR Outlier Thresholds:")
print(f"Lower Bound: {lower_bound} minutes")
print(f"Upper Bound: {upper_bound} minutes")
print(f"Number of outliers found: {len(outliers)}")

# Display the top 10 longest trips
print("\n--- Top 10 Longest Trips ---")
display(tables["trips"].nlargest(10, "duration_min"))

--- Summary Statistics ---
count     570.000000
mean       28.054386
std       146.569161
min         4.000000
25%        12.000000
50%        16.000000
75%        22.000000
max      2880.000000
Name: duration_min, dtype: float64

99th Percentile: 51.309999999999945 minutes

IQR Outlier Thresholds:
Lower Bound: -3.0 minutes
Upper Bound: 37.0 minutes
Number of outliers found: 19

--- Top 10 Longest Trips ---


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type,user_type_clean
80,T0081,B010,S12,S11,2025-03-04 21:57:00,2880.0,Member,Member
143,T0143,B020,S09,S10,2025-03-08 12:23:00,1440.0,Casual,Casual
287,T0284,B046,S08,S03,2025-03-15 20:25:00,1440.0,Member,Member
127,T0127,B025,S08,S09,2025-03-07 09:01:00,90.0,Member,Member
357,T0352,B042,S03,S02,2025-03-19 08:59:00,67.0,Member,Member
326,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member,Member
520,T0512,B045,S07,S04,26/03/2025 18:32,51.0,Member,Member
37,T0038,B016,S12,S09,2025-03-02 12:23:00,50.0,Casual,Casual
262,T0260,B025,S04,S04,15/03/2025 07:29,46.0,Member,Member
268,T0266,B046,S11,S01,15/03/2025 13:53,46.0,Member,Member


## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

## 6. What I could not verify

*What remains unchecked? What would you do next?*